Jędrzej Czarnota, Lab Week 1 Day 2

In [3]:
import random as rd
import datetime as dt

In [4]:
def create_sensors():
    """Returns a list of the 4 home sensors based on the initial setup."""
    return [
        {
            "id": 1,
            "location": "Living Room",
            "type": "motion",
            "current_value": None
        },
        {
            "id": 2,
            "location": "Front Door",
            "type": "door",
            "current_value": None
        },
        {
            "id": 3,
            "location": "Kitchen",
            "type": "temperature",
            "current_value": None
        },
        {
            "id": 4,
            "location": "Bedroom",
            "type": "smoke",
            "current_value": None
        }
    ]

def create_alert(time, severity, category, message):
    """Creates a dictionary representing an alert event."""
    return {
        "time": time,
        "severity": severity,
        "category": category,
        "message": message
    }

# Set the initial mode for the system
system_mode = "AWAY"

# Check: Create the sensors and print them out
my_sensors = create_sensors()

print(f"Current System Mode: {system_mode}\n")
print("Initialized Sensors:")
for sensor in my_sensors:
    print(sensor)

Current System Mode: AWAY

Initialized Sensors:
{'id': 1, 'location': 'Living Room', 'type': 'motion', 'current_value': None}
{'id': 2, 'location': 'Front Door', 'type': 'door', 'current_value': None}
{'id': 3, 'location': 'Kitchen', 'type': 'temperature', 'current_value': None}
{'id': 4, 'location': 'Bedroom', 'type': 'smoke', 'current_value': None}


In [6]:
from datetime import datetime

# --- Constants ---
FREEZE_TEMP = 35
HEAT_TEMP = 95
COMFORT_MIN = 65
COMFORT_MAX = 75

def check_sensor_reading(sensor, mode):
    """
    Evaluates a sensor's current value against the system mode.
    Returns an alert dictionary if a condition is met, otherwise returns None.
    """
    val = sensor["current_value"]
    s_type = sensor["type"]
    location = sensor["location"]
    
    # If there's no reading yet, everything is fine
    if val is None:
        return None

    # Get a timestamp for the alert
    current_time = datetime.now().strftime("%H:%M:%S")

    # 1. Smoke Rule (Always CRITICAL)
    if s_type == "smoke" and val is True:
        return create_alert(current_time, "CRITICAL", "SAFETY", f"Smoke detected in the {location}!")

    # 2. Temperature Rules (SAFETY vs COMFORT)
    if s_type == "temperature":
        if val < FREEZE_TEMP or val > HEAT_TEMP:
            return create_alert(current_time, "HIGH", "SAFETY", f"{location} temperature is {val}°F (Dangerous)!")
        elif mode == "HOME" and (val < COMFORT_MIN or val > COMFORT_MAX):
            return create_alert(current_time, "LOW", "COMFORT", f"{location} temperature is {val}°F (Uncomfortable).")

    # 3. Motion & Door Rules (Depends heavily on mode)
    # Assuming 'val' is True for motion detected or "OPEN" for doors
    is_active = (val is True) or (val == "OPEN") or (val == "OPENED")
    
    if s_type in ["motion", "door"] and is_active:
        if mode == "AWAY":
            return create_alert(current_time, "HIGH", "SECURITY", f"{location} {s_type} activated while in AWAY mode!")
        elif mode == "SLEEP":
            return create_alert(current_time, "MEDIUM", "SECURITY", f"{location} {s_type} activated while in SLEEP mode!")
        # If mode is HOME, it returns None (doing nothing, as per the table)

    return None

# --- Check (Manual Test) ---

# Re-defining create_alert here just in case it's a new cell
def create_alert(time, severity, category, message):
    return {"time": time, "severity": severity, "category": category, "message": message}

# Create a dummy door sensor
test_door_sensor = {
    "id": 2,
    "location": "Front Door",
    "type": "door",
    "current_value": "OPENED"
}

# Test 1: AWAY mode (Should trigger HIGH alert)
away_result = check_sensor_reading(test_door_sensor, "AWAY")
print("Test 1 (AWAY):", away_result)

# Test 2: HOME mode (Should return None)
home_result = check_sensor_reading(test_door_sensor, "HOME")
print("Test 2 (HOME):", home_result)

Test 1 (AWAY): {'time': '16:50:50', 'severity': 'HIGH', 'category': 'SECURITY', 'message': 'Front Door door activated while in AWAY mode!'}
Test 2 (HOME): None


In [7]:
import random

# Global list to store logged alerts
log_history = []


def generate_reading(sensor):
    """Generates a realistic random reading and updates sensor['current_value']."""
    s_type = sensor["type"]

    if s_type == "motion":
        # True ~20% of the time
        sensor["current_value"] = random.random() < 0.2
    elif s_type == "door":
        # Mostly closed, occasionally opened
        sensor["current_value"] = (
            "OPENED" if random.random() < 0.25 else "CLOSED"
        )
    elif s_type == "temperature":
        # Random temperature between 30°F and 100°F
        sensor["current_value"] = random.randint(30, 100)
    elif s_type == "smoke":
        # Smoke is rare: True ~5% of the time
        sensor["current_value"] = random.random() < 0.05

    return sensor["current_value"]


def trigger_alert(alert):
    """Prints the formatted alert message with an appropriate icon."""
    severity_icons = {
        "CRITICAL": "🔥",
        "HIGH": "🚨",
        "MEDIUM": "⚠️",
        "LOW": "💡",
    }
    icon = severity_icons.get(alert["severity"], "⚠️")
    print(
        f"[ALERT!] {icon} {alert['severity']}: {alert['category']}: {alert['message']}"
    )


def log_event(alert, logs):
    """Prints a log message and appends the alert entry to the history list."""
    log_entry = f"[{alert['time']}] Alert sent: {alert['message']}"
    logs.append(log_entry)
    print(f"[LOG] [{alert['time']}] Sending notification to homeowner...")


def process_readings(sensors, mode, logs):
    """Generates readings for all sensors, displays them, and handles any triggered alerts."""
    for sensor in sensors:
        val = generate_reading(sensor)
        s_type = sensor["type"]
        loc = sensor["location"]

        # Display formatted sensor reading
        if s_type == "motion":
            display_val = "Motion detected" if val else "No activity"
            print(f"[READING] {loc} Motion: {display_val}")
        elif s_type == "door":
            print(f"[READING] {loc}: {val}")
        elif s_type == "temperature":
            status = (
                "Normal"
                if (COMFORT_MIN <= val <= COFORT_MAX
                    if "COFORT_MAX" in locals()
                    else 65 <= val <= 75)
                else "Unusual"
            )
            print(f"[READING] {loc} Temperature: {val}°F ({status})")
        elif s_type == "smoke":
            display_val = "SMOKE DETECTED" if val else "CLEAR"
            print(f"[READING] {loc} Smoke: {display_val}")

        # Check reading against rules
        alert = check_sensor_reading(sensor, mode)
        if alert:
            trigger_alert(alert)
            log_event(alert, logs)


# --- Check / Demonstration ---
sensors = create_sensors()
system_mode = "AWAY"

print("=== Running a Simulation Cycle ===")
process_readings(sensors, system_mode, log_history)

print("\n--- Current Log History ---")
for entry in log_history:
    print(entry)

=== Running a Simulation Cycle ===
[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 35°F (Unusual)
[READING] Bedroom Smoke: CLEAR

--- Current Log History ---


In [8]:
# 1. Reset sensors and log histories for each mode
sensors = create_sensors()
logs_away = []
logs_home = []

# 2. Generate ONE fixed set of readings for the comparison
print("=== Current Sensor Readings (Shared) ===")
for sensor in sensors:
    generate_reading(sensor)
    print(
        f"• {sensor['location']} ({sensor['type']}): {sensor['current_value']}"
    )

print("\n" + "=" * 45)

# 3. Test under AWAY mode
print("--- Evaluation: AWAY Mode ---")
for sensor in sensors:
    alert = check_sensor_reading(sensor, "AWAY")
    if alert:
        trigger_alert(alert)
        log_event(alert, logs_away)

if not logs_away:
    print("No alerts triggered in AWAY mode.")

print("\n" + "=" * 45)

# 4. Test under HOME mode
print("--- Evaluation: HOME Mode ---")
for sensor in sensors:
    alert = check_sensor_reading(sensor, "HOME")
    if alert:
        trigger_alert(alert)
        log_event(alert, logs_home)

if not logs_home:
    print("No alerts triggered in HOME mode.")

print("\n" + "=" * 45)

# 5. Summary comparison
print("=== Summary Comparison ===")
print(f"Total alerts in AWAY mode: {len(logs_away)}")
print(f"Total alerts in HOME mode: {len(logs_home)}")

=== Current Sensor Readings (Shared) ===
• Living Room (motion): False
• Front Door (door): CLOSED
• Kitchen (temperature): 91
• Bedroom (smoke): False

--- Evaluation: AWAY Mode ---
No alerts triggered in AWAY mode.

--- Evaluation: HOME Mode ---
[ALERT!] 💡 LOW: COMFORT: Kitchen temperature is 91°F (Uncomfortable).
[LOG] [16:55:09] Sending notification to homeowner...

=== Summary Comparison ===
Total alerts in AWAY mode: 0
Total alerts in HOME mode: 1


In [9]:
import random
from datetime import datetime

# --- Constants ---
FREEZE_TEMP = 35
HEAT_TEMP = 95
COMFORT_MIN = 65
COMFORT_MAX = 75


class Sensor:

    def __init__(self, sensor_id, location, sensor_type):
        """Initializes a sensor object with an ID, location, type, and empty value."""
        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.current_value = None

    def read(self):
        """Generates a random reading according to the sensor type and stores it."""
        if self.type == "motion":
            self.current_value = random.random() < 0.2
        elif self.type == "door":
            self.current_value = (
                "OPENED" if random.random() < 0.25 else "CLOSED"
            )
        elif self.type == "temperature":
            self.current_value = random.randint(30, 100)
        elif self.type == "smoke":
            self.current_value = random.random() < 0.05

        return self.current_value

    def reset(self):
        """Resets the current sensor reading back to None."""
        self.current_value = None

    def isAbnormal(self, mode=None):
        """Checks whether the current reading is abnormal.

        If a mode is provided, checks if it triggers an alert under that mode.
        If no mode is provided, checks if the reading is inherently outside
        normal baselines.
        """
        val = self.current_value
        if val is None:
            return False

        # Mode-specific evaluation
        if mode is not None:
            if self.type == "smoke" and val is True:
                return True
            if self.type == "temperature":
                if val < FREEZE_TEMP or val > HEAT_TEMP:
                    return True
                if mode == "HOME" and (val < COMFORT_MIN or val > COMFORT_MAX):
                    return True
            if self.type in ["motion", "door"]:
                is_active = (
                    (val is True) or (val == "OPENED") or (val == "OPEN")
                )
                if is_active and mode in ["AWAY", "SLEEP"]:
                    return True
            return False

        # Standalone sensor baseline evaluation (no mode provided)
        if self.type == "motion":
            return val is True
        elif self.type == "door":
            return val in ["OPEN", "OPENED"]
        elif self.type == "smoke":
            return val is True
        elif self.type == "temperature":
            return val < COMFORT_MIN or val > COMFORT_MAX

        return False

In [10]:
# Create one sensor, read it, and inspect its attributes
single_sensor = Sensor(sensor_id=1, location="Living Room", sensor_type="motion")
single_sensor.read()

print(f"Sensor ID:     {single_sensor.id}")
print(f"Location:      {single_sensor.location}")
print(f"Type:          {single_sensor.type}")
print(f"Current Value: {single_sensor.current_value}")
print(f"Is Abnormal:   {single_sensor.isAbnormal()}")

# Test reset
single_sensor.reset()
print(f"After Reset:   {single_sensor.current_value}")

Sensor ID:     1
Location:      Living Room
Type:          motion
Current Value: False
Is Abnormal:   False
After Reset:   None


In [11]:
def create_sensors():
    """Instantiates the 4 system sensors as Sensor objects."""
    return [
        Sensor(1, "Living Room", "motion"),
        Sensor(2, "Front Door", "door"),
        Sensor(3, "Kitchen", "temperature"),
        Sensor(4, "Bedroom", "smoke"),
    ]


def check_sensor_reading(sensor, mode):
    """Uses the Sensor object to create an alert if isAbnormal returns True."""
    val = sensor.current_value
    if val is None or not sensor.isAbnormal(mode):
        return None

    current_time = datetime.now().strftime("%H:%M:%S")

    # Critical smoke alert
    if sensor.type == "smoke":
        return create_alert(
            current_time,
            "CRITICAL",
            "SAFETY",
            f"Smoke detected in the {sensor.location}!",
        )

    # Temperature alerts
    if sensor.type == "temperature":
        if val < FREEZE_TEMP or val > HEAT_TEMP:
            return create_alert(
                current_time,
                "HIGH",
                "SAFETY",
                f"{sensor.location} temperature is {val}°F (Dangerous)!",
            )
        elif mode == "HOME":
            return create_alert(
                current_time,
                "LOW",
                "COMFORT",
                f"{sensor.location} temperature is {val}°F (Uncomfortable).",
            )

    # Security alerts
    if sensor.type in ["motion", "door"]:
        severity = "HIGH" if mode == "AWAY" else "MEDIUM"
        return create_alert(
            current_time,
            severity,
            "SECURITY",
            f"{sensor.location} {sensor.type} activated while in {mode} mode!",
        )

    return None


def process_readings(sensors, mode, logs):
    """Calls .read() directly on each Sensor object and checks alerts."""
    for sensor in sensors:
        val = sensor.read()

        # Display output
        if sensor.type == "motion":
            display = "Motion detected" if val else "No activity"
            print(f"[READING] {sensor.location} Motion: {display}")
        elif sensor.type == "door":
            print(f"[READING] {sensor.location}: {val}")
        elif sensor.type == "temperature":
            status = "Normal" if COMFORT_MIN <= val <= COMFORT_MAX else "Unusual"
            print(
                f"[READING] {sensor.location} Temperature: {val}°F ({status})"
            )
        elif sensor.type == "smoke":
            display = "SMOKE DETECTED" if val else "CLEAR"
            print(f"[READING] {sensor.location} Smoke: {display}")

        # Check for alert
        alert = check_sensor_reading(sensor, mode)
        if alert:
            trigger_alert(alert)
            log_event(alert, logs)


# --- Run Complete Object-Oriented Simulation ---
house_sensors = create_sensors()
logs = []

print("=== Running with Sensor Objects (AWAY Mode) ===")
process_readings(house_sensors, "AWAY", logs)

=== Running with Sensor Objects (AWAY Mode) ===
[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 90°F (Unusual)
[READING] Bedroom Smoke: CLEAR


In [12]:
from datetime import datetime, timedelta
import random

# --- Constants ---
FREEZE_TEMP = 35
HEAT_TEMP = 95
COMFORT_MIN = 65
COMFORT_MAX = 75

# --- Sensor Class ---
class Sensor:
    def __init__(self, sensor_id, location, sensor_type):
        self.id = sensor_id
        self.location = location
        self.type = sensor_type
        self.current_value = None

    def read(self):
        """Generates realistic sensor readings."""
        if self.type == "motion":
            self.current_value = random.random() < 0.35  # ~35% chance of motion
        elif self.type == "door":
            self.current_value = "OPENED" if random.random() < 0.30 else "CLOSED"
        elif self.type == "temperature":
            self.current_value = random.randint(30, 100)
        elif self.type == "smoke":
            self.current_value = random.random() < 0.10  # ~10% chance of smoke
        return self.current_value

    def reset(self):
        self.current_value = None

    def isAbnormal(self, mode):
        val = self.current_value
        if val is None:
            return False

        if self.type == "smoke" and val is True:
            return True
        if self.type == "temperature":
            if val < FREEZE_TEMP or val > HEAT_TEMP:
                return True
            if mode == "HOME" and (val < COMFORT_MIN or val > COMFORT_MAX):
                return True
        if self.type in ["motion", "door"]:
            is_active = (val is True) or (val in ["OPEN", "OPENED"])
            if is_active and mode in ["AWAY", "SLEEP"]:
                return True
        return False

# --- Helper Functions ---
def create_alert(time_str, severity, category, message):
    return {
        "time": time_str,
        "severity": severity,
        "category": category,
        "message": message
    }

def check_sensor_reading(sensor, mode, current_time):
    if not sensor.isAbnormal(mode):
        return None

    val = sensor.current_value

    # 1. Smoke (Safety - CRITICAL)
    if sensor.type == "smoke":
        return create_alert(current_time, "CRITICAL", "SAFETY", f"Smoke detected in {sensor.location}!")

    # 2. Temperature (Safety vs Comfort)
    if sensor.type == "temperature":
        if val < FREEZE_TEMP or val > HEAT_TEMP:
            return create_alert(current_time, "HIGH", "SAFETY", f"{sensor.location} temperature is {val}°F (Hazard)!")
        elif mode == "HOME":
            return create_alert(current_time, "LOW", "COMFORT", f"{sensor.location} temperature is {val}°F (Uncomfortable).")

    # 3. Security (Door / Motion)
    if sensor.type == "door":
        return create_alert(current_time, "HIGH", "SECURITY", f"{sensor.location} opened while in {mode} mode!")
    if sensor.type == "motion":
        return create_alert(current_time, "HIGH", "SECURITY", f"Motion detected in {sensor.location} while in {mode} mode!")

    return None

def trigger_alert(alert):
    severity_icons = {
        "CRITICAL": "🔥",
        "HIGH": "🚨",
        "MEDIUM": "⚠️",
        "LOW": "💡"
    }
    icon = severity_icons.get(alert["severity"], "⚠️")
    print(f"[ALERT!] {icon} {alert['severity']}: {alert['category']}: {alert['message']}")

def log_event(alert, log_store):
    log_store.append(f"[{alert['time']}] {alert['severity']}: {alert['message']}")
    print(f"[LOG] [{alert['time']}] Sending notification to homeowner...")

def format_reading(sensor):
    val = sensor.current_value
    if sensor.type == "motion":
        desc = "Motion detected" if val else "No activity"
        return f"[READING] {sensor.location} Motion: {desc}"
    elif sensor.type == "door":
        return f"[READING] {sensor.location}: {val}"
    elif sensor.type == "temperature":
        desc = "Normal" if COMFORT_MIN <= val <= COMFORT_MAX else "Unusual"
        return f"[READING] {sensor.location} Temperature: {val}°F ({desc})"
    elif sensor.type == "smoke":
        desc = "SMOKE DETECTED" if val else "CLEAR"
        return f"[READING] {sensor.location} Smoke: {desc}"

In [13]:
# --- Main Simulation Loop ---
def run_simulation(mode="AWAY", minutes=3, start_clock="14:30:00"):
    sensors = [
        Sensor(1, "Living Room", "motion"),
        Sensor(2, "Front Door", "door"),
        Sensor(3, "Kitchen", "temperature"),
        Sensor(4, "Bedroom", "smoke")
    ]
    
    current_dt = datetime.strptime(start_clock, "%H:%M:%S")
    logs = []

    print("=== HomeGuard Security System ===")
    
    for cycle in range(minutes):
        time_str = current_dt.strftime("%H:%M:%S")
        
        if cycle == 0:
            print(f"Time: {time_str}")
            print(f"Mode: {mode}\n")
        else:
            print(f"\nTime: {time_str}")

        # Read and evaluate all sensors
        for sensor in sensors:
            sensor.read()
            print(format_reading(sensor))
            
            alert = check_sensor_reading(sensor, mode, time_str)
            if alert:
                trigger_alert(alert)
                log_event(alert, logs)
        
        # Advance time by 1 minute
        current_dt += timedelta(minutes=1)

    return logs

# Execute the 3-minute simulation
run_simulation(mode="AWAY", minutes=3)

=== HomeGuard Security System ===
Time: 14:30:00
Mode: AWAY

[READING] Living Room Motion: Motion detected
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!
[LOG] [14:30:00] Sending notification to homeowner...
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 94°F (Unusual)
[READING] Bedroom Smoke: CLEAR

Time: 14:31:00
[READING] Living Room Motion: Motion detected
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!
[LOG] [14:31:00] Sending notification to homeowner...
[READING] Front Door: OPENED
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [14:31:00] Sending notification to homeowner...
[READING] Kitchen Temperature: 77°F (Unusual)
[READING] Bedroom Smoke: CLEAR

Time: 14:32:00
[READING] Living Room Motion: Motion detected
[ALERT!] 🚨 HIGH: SECURITY: Motion detected in Living Room while in AWAY mode!
[LOG] [14:32:00] Sending notification to homeowner...
[READING] Front Door: CLOSED
[READING] Kit

['[14:30:00] HIGH: Motion detected in Living Room while in AWAY mode!',
 '[14:31:00] HIGH: Motion detected in Living Room while in AWAY mode!',
 '[14:31:00] HIGH: Front Door opened while in AWAY mode!',
 '[14:32:00] HIGH: Motion detected in Living Room while in AWAY mode!',
 '[14:32:00] CRITICAL: Smoke detected in Bedroom!']

In [14]:
print("==========================================")
print("TEST 1: SECURITY (AWAY Mode)")
print("==========================================")
run_simulation(mode="AWAY", minutes=3)

TEST 1: SECURITY (AWAY Mode)
=== HomeGuard Security System ===
Time: 14:30:00
Mode: AWAY

[READING] Living Room Motion: No activity
[READING] Front Door: OPENED
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [14:30:00] Sending notification to homeowner...
[READING] Kitchen Temperature: 42°F (Unusual)
[READING] Bedroom Smoke: CLEAR

Time: 14:31:00
[READING] Living Room Motion: No activity
[READING] Front Door: OPENED
[ALERT!] 🚨 HIGH: SECURITY: Front Door opened while in AWAY mode!
[LOG] [14:31:00] Sending notification to homeowner...
[READING] Kitchen Temperature: 98°F (Unusual)
[ALERT!] 🚨 HIGH: SAFETY: Kitchen temperature is 98°F (Hazard)!
[LOG] [14:31:00] Sending notification to homeowner...
[READING] Bedroom Smoke: CLEAR

Time: 14:32:00
[READING] Living Room Motion: No activity
[READING] Front Door: CLOSED
[READING] Kitchen Temperature: 81°F (Unusual)
[READING] Bedroom Smoke: CLEAR


['[14:30:00] HIGH: Front Door opened while in AWAY mode!',
 '[14:31:00] HIGH: Front Door opened while in AWAY mode!',
 '[14:31:00] HIGH: Kitchen temperature is 98°F (Hazard)!']

In [15]:
print("==========================================")
print("TEST 2: COMFORT (HOME Mode)")
print("==========================================")
run_simulation(mode="HOME", minutes=3)

TEST 2: COMFORT (HOME Mode)
=== HomeGuard Security System ===
Time: 14:30:00
Mode: HOME

[READING] Living Room Motion: Motion detected
[READING] Front Door: OPENED
[READING] Kitchen Temperature: 71°F (Normal)
[READING] Bedroom Smoke: CLEAR

Time: 14:31:00
[READING] Living Room Motion: Motion detected
[READING] Front Door: OPENED
[READING] Kitchen Temperature: 100°F (Unusual)
[ALERT!] 🚨 HIGH: SAFETY: Kitchen temperature is 100°F (Hazard)!
[LOG] [14:31:00] Sending notification to homeowner...
[READING] Bedroom Smoke: SMOKE DETECTED
[ALERT!] 🔥 CRITICAL: SAFETY: Smoke detected in Bedroom!
[LOG] [14:31:00] Sending notification to homeowner...

Time: 14:32:00
[READING] Living Room Motion: No activity
[READING] Front Door: OPENED
[READING] Kitchen Temperature: 78°F (Unusual)
[ALERT!] 💡 LOW: COMFORT: Kitchen temperature is 78°F (Uncomfortable).
[LOG] [14:32:00] Sending notification to homeowner...
[READING] Bedroom Smoke: CLEAR


['[14:31:00] HIGH: Kitchen temperature is 100°F (Hazard)!',
 '[14:31:00] CRITICAL: Smoke detected in Bedroom!',
 '[14:32:00] LOW: Kitchen temperature is 78°F (Uncomfortable).']

In [16]:
print("==========================================")
print("TEST 3: SAFETY (Forced Critical Conditions)")
print("==========================================")

safety_sensors = [
    Sensor(3, "Kitchen", "temperature"),
    Sensor(4, "Bedroom", "smoke")
]

# Set critical hazards manually
safety_sensors[0].current_value = 28   # Below FREEZE_TEMP (35°F)
safety_sensors[1].current_value = True # Smoke active

for test_mode in ["AWAY", "HOME"]:
    print(f"\nEvaluating Safety under Mode: {test_mode}")
    for sensor in safety_sensors:
        print(format_reading(sensor))
        alert = check_sensor_reading(sensor, test_mode, "12:00:00")
        if alert:
            trigger_alert(alert)

TEST 3: SAFETY (Forced Critical Conditions)

Evaluating Safety under Mode: AWAY
[READING] Kitchen Temperature: 28°F (Unusual)
[ALERT!] 🚨 HIGH: SAFETY: Kitchen temperature is 28°F (Hazard)!
[READING] Bedroom Smoke: SMOKE DETECTED
[ALERT!] 🔥 CRITICAL: SAFETY: Smoke detected in Bedroom!

Evaluating Safety under Mode: HOME
[READING] Kitchen Temperature: 28°F (Unusual)
[ALERT!] 🚨 HIGH: SAFETY: Kitchen temperature is 28°F (Hazard)!
[READING] Bedroom Smoke: SMOKE DETECTED
[ALERT!] 🔥 CRITICAL: SAFETY: Smoke detected in Bedroom!


# Lab Proof: HomeGuard Security System

## Program Path
`./homeguard.py`

## Run Command
```bash
python3 homeguard.py

Edge Case Explanation: Strict Inequality at Freezing Threshold (35°F)
Scenario: The kitchen temperature sensor registers exactly 35°F.
Explanation:
The system rule defines a safety hazard as temperature strictly below 35°F (val < 35) or above 95°F (val > 95). When the kitchen reading hits exactly 35°F:
Missed Hazard Threshold: Because 35 is not strictly less than 35, the safety condition evaluates to False. The system fails to register a HIGH: SAFETY freeze warning.
Mode-Dependent Misclassification:
In HOME mode, the reading falls below the comfort baseline (65°F), causing the system to downgrade a near-freezing pipe risk into a minor 💡 LOW: COMFORT notification.
In AWAY mode, comfort alerts are ignored entirely, meaning the 35°F temperature produces zero alerts despite pipes being at risk.
Resolution: Update the condition from strict inequality (<) to inclusive inequality (val <= FREEZE_TEMP) so that threshold boundary temperatures trigger immediate safety alerts.